### AutoCatalogAI — Reproducible Model Comparison

This notebook creates a fair comparison between:

1. **Majority Baseline**
2. **Frozen CLIP + Multi-task Heads (V1)**
3. **AutoCatalogAI V2 (fine-tuned production model)**

In [ ]:
%pip uninstall -y torch torchvision torchaudio

%pip install -q --no-cache-dir \
    torch==2.5.1 \
    torchvision==0.20.1 \
    torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu121

%pip install -q \
    transformers==4.46.3 \
    datasets==3.1.0 \
    huggingface-hub==0.26.2 \
    scikit-learn==1.5.2 \
    pandas==2.2.3 \
    numpy==1.26.4 \
    Pillow==11.0.0 \
    tqdm==4.67.1

## 2. Imports and Configuration

In [1]:
import gc
import hashlib
import json
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import CLIPImageProcessor, CLIPModel

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)


Python: 3.12.13
Torch: 2.5.1+cu121
Transformers: 4.46.3
CUDA available: True
GPU: Tesla P100-PCIE-16GB
CUDA runtime: 12.1


In [ ]:
DATASET_NAME = "ashraq/fashion-product-images-small"
V1_REPO_ID = "mohsin416/autocatalogai-clip-multitask"
V2_REPO_ID = "mohsin416/autocatalogai-clip-multitask-v2"

TASKS = [
    "gender",
    "masterCategory",
    "subCategory",
    "articleType",
    "baseColour",
    "season",
    "usage",
]

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

BATCH_SIZE = 64
NUM_WORKERS = 0

LATENCY_WARMUP_RUNS = 20
LATENCY_MEASURED_RUNS = 100

COLOR_IMAGE_SIZE = 128
COLOR_FEATURE_DIM = 37

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT_DIR = Path(".")
OUTPUT_DIR = ROOT_DIR / "artifacts" / "evaluation" / "model_comparison"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 3. Reproducibility

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


set_seed(SEED)


## 4. Download Published Model Metadata

The comparison uses the exact published V1 and V2 checkpoints.  
V1 represents the frozen-CLIP baseline. V2 represents the production model.


In [4]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=True,
        )
    except (TypeError, RuntimeError):
        return torch.load(
            path,
            map_location=map_location,
        )


def download_repo_artifacts(repo_id, include_rules=False):
    filenames = [
        "model.pt",
        "config.json",
        "label_maps.json",
    ]

    if include_rules:
        filenames.append("consistency_rules.json")

    paths = {
        filename: hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            repo_type="model",
        )
        for filename in filenames
    }

    artifacts = {
        "checkpoint": safe_torch_load(paths["model.pt"]),
        "config": load_json(paths["config.json"]),
        "label_maps": load_json(paths["label_maps.json"]),
    }

    if include_rules:
        artifacts["consistency_rules"] = load_json(
            paths["consistency_rules.json"]
        )

    return artifacts


v1_artifacts = download_repo_artifacts(V1_REPO_ID)
v2_artifacts = download_repo_artifacts(
    V2_REPO_ID,
    include_rules=True,
)

if v1_artifacts["label_maps"] != v2_artifacts["label_maps"]:
    raise ValueError(
        "V1 and V2 label maps are different. "
        "A direct comparison would not be valid."
    )

label_maps = v2_artifacts["label_maps"]
v1_config = v1_artifacts["config"]
v2_config = v2_artifacts["config"]

MODEL_NAME = (
    v2_config.get("base_model_name")
    or v2_config.get("model_name")
    or "openai/clip-vit-base-patch32"
)

HIDDEN_DIM = int(v2_config.get("hidden_dim", 512))
DROPOUT = float(v2_config.get("dropout", 0.2))

task_num_classes = {
    task: len(label_maps[task]["label2id"])
    for task in TASKS
}

print("Base model:", MODEL_NAME)
print("Task classes:", task_num_classes)


Base model: openai/clip-vit-base-patch32
Task classes: {'gender': 5, 'masterCategory': 7, 'subCategory': 45, 'articleType': 141, 'baseColour': 46, 'season': 4, 'usage': 8}


## 5. Load and Clean the Dataset

The same validation rules used in the training notebook are applied here.


In [5]:
raw_dataset = load_dataset(
    DATASET_NAME,
    split="train",
)

missing_columns = [
    task
    for task in TASKS
    if task not in raw_dataset.column_names
]

if "image" not in raw_dataset.column_names:
    raise ValueError("Dataset must contain an image column.")

if missing_columns:
    raise ValueError(
        f"Dataset is missing task columns: {missing_columns}"
    )


def is_valid_row(row):
    if row.get("image") is None:
        return False

    for task in TASKS:
        value = row.get(task)

        if value is None:
            return False

        value = str(value).strip()

        if not value:
            return False

        if value not in label_maps[task]["label2id"]:
            return False

    return True


clean_dataset = raw_dataset.filter(is_valid_row)

print("Raw samples:", len(raw_dataset))
print("Clean samples:", len(clean_dataset))
print("Dataset fingerprint:", clean_dataset._fingerprint)


Filter:   0%|          | 0/44072 [00:00<?, ? examples/s]

Raw samples: 44072
Clean samples: 44072
Dataset fingerprint: 5cacd020bfdb9ce5


## 6. Recreate the Exact 70/15/15 Split

The split uses seed `42` and article-type stratification, matching the V2 training notebook.


In [6]:
metadata = {
    task: [
        str(value).strip()
        for value in clean_dataset[task]
    ]
    for task in TASKS
}

df = pd.DataFrame(metadata)
df["dataset_idx"] = np.arange(len(clean_dataset))


def make_safe_stratify_labels(series):
    counts = series.value_counts()

    return series.apply(
        lambda value: (
            value
            if counts[value] >= 2
            else "__rare__"
        )
    )


all_indices = df.index.to_numpy()

try:
    train_idx, temporary_idx = train_test_split(
        all_indices,
        test_size=VAL_RATIO + TEST_RATIO,
        random_state=SEED,
        stratify=make_safe_stratify_labels(
            df["articleType"]
        ),
    )
except ValueError:
    train_idx, temporary_idx = train_test_split(
        all_indices,
        test_size=VAL_RATIO + TEST_RATIO,
        random_state=SEED,
    )

temporary_df = df.loc[temporary_idx]

try:
    val_idx, test_idx = train_test_split(
        temporary_idx,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        random_state=SEED,
        stratify=make_safe_stratify_labels(
            temporary_df["articleType"]
        ),
    )
except ValueError:
    val_idx, test_idx = train_test_split(
        temporary_idx,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        random_state=SEED,
    )

train_df = df.loc[train_idx].copy()
val_df = df.loc[val_idx].copy()
test_df = df.loc[test_idx].copy()

assert set(train_idx).isdisjoint(val_idx)
assert set(train_idx).isdisjoint(test_idx)
assert set(val_idx).isdisjoint(test_idx)

train_df.to_csv(
    PROCESSED_DIR / "train_v2.csv",
    index=False,
)
val_df.to_csv(
    PROCESSED_DIR / "val_v2.csv",
    index=False,
)
test_df.to_csv(
    PROCESSED_DIR / "test_v2.csv",
    index=False,
)

test_index_bytes = np.asarray(
    sorted(test_idx),
    dtype=np.int64,
).tobytes()

test_split_sha256 = hashlib.sha256(
    test_index_bytes
).hexdigest()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Test split SHA256:", test_split_sha256)


Train: 30850
Validation: 6611
Test: 6611
Test split SHA256: 106737acc60a436248d35e8a375a014d8b6e2f26f74b147b6444540b7781c0ec


## 7. Test Dataset and DataLoader

In [7]:
processor = CLIPImageProcessor.from_pretrained(
    MODEL_NAME
)


def extract_color_features(
    image,
    image_size=COLOR_IMAGE_SIZE,
):
    image = image.convert("RGB").resize(
        (image_size, image_size)
    )

    margin = int(image_size * 0.10)

    image = image.crop(
        (
            margin,
            margin,
            image_size - margin,
            image_size - margin,
        )
    )

    rgb = np.asarray(
        image,
        dtype=np.float32,
    ) / 255.0

    hsv = np.asarray(
        image.convert("HSV"),
        dtype=np.float32,
    ) / 255.0

    rgb_flat = rgb.reshape(-1, 3)
    hsv_flat = hsv.reshape(-1, 3)

    saturation = hsv_flat[:, 1]
    value = hsv_flat[:, 2]

    foreground_mask = (
        (saturation > 0.08)
        | (value < 0.92)
    )

    if foreground_mask.sum() < 256:
        foreground_mask = np.ones(
            len(hsv_flat),
            dtype=bool,
        )

    selected_rgb = rgb_flat[foreground_mask]
    selected_hsv = hsv_flat[foreground_mask]

    hue_hist, _ = np.histogram(
        selected_hsv[:, 0],
        bins=12,
        range=(0.0, 1.0),
    )

    saturation_hist, _ = np.histogram(
        selected_hsv[:, 1],
        bins=8,
        range=(0.0, 1.0),
    )

    value_hist, _ = np.histogram(
        selected_hsv[:, 2],
        bins=8,
        range=(0.0, 1.0),
    )

    hue_hist = hue_hist.astype(np.float32)
    saturation_hist = saturation_hist.astype(np.float32)
    value_hist = value_hist.astype(np.float32)

    hue_hist /= max(hue_hist.sum(), 1.0)
    saturation_hist /= max(
        saturation_hist.sum(),
        1.0,
    )
    value_hist /= max(value_hist.sum(), 1.0)

    rgb_mean = selected_rgb.mean(
        axis=0
    ).astype(np.float32)

    rgb_std = selected_rgb.std(
        axis=0
    ).astype(np.float32)

    rgb_median = np.median(
        selected_rgb,
        axis=0,
    ).astype(np.float32)

    features = np.concatenate(
        [
            hue_hist,
            saturation_hist,
            value_hist,
            rgb_mean,
            rgb_std,
            rgb_median,
        ]
    ).astype(np.float32)

    if features.shape[0] != COLOR_FEATURE_DIM:
        raise ValueError(
            f"Expected {COLOR_FEATURE_DIM} color features, "
            f"got {features.shape[0]}"
        )

    return features


class ComparisonDataset(Dataset):
    def __init__(
        self,
        source_dataset,
        indices,
        processor,
        label_maps,
    ):
        self.source_dataset = source_dataset
        self.indices = list(map(int, indices))
        self.processor = processor
        self.label_maps = label_maps

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        global_index = self.indices[index]
        item = self.source_dataset[global_index]

        image = item["image"]

        if not isinstance(image, Image.Image):
            image = Image.open(image)

        image = image.convert("RGB")

        pixel_values = self.processor(
            images=image,
            return_tensors="pt",
        )["pixel_values"].squeeze(0)

        color_features = torch.tensor(
            extract_color_features(image),
            dtype=torch.float32,
        )

        labels = {
            task: torch.tensor(
                self.label_maps[task]["label2id"][
                    str(item[task]).strip()
                ],
                dtype=torch.long,
            )
            for task in TASKS
        }

        return {
            "pixel_values": pixel_values,
            "color_features": color_features,
            "labels": labels,
            "global_index": global_index,
        }


def collate_batch(batch):
    return {
        "pixel_values": torch.stack(
            [item["pixel_values"] for item in batch]
        ),
        "color_features": torch.stack(
            [item["color_features"] for item in batch]
        ),
        "labels": {
            task: torch.stack(
                [item["labels"][task] for item in batch]
            )
            for task in TASKS
        },
        "global_indices": [
            item["global_index"]
            for item in batch
        ],
    }


test_dataset = ComparisonDataset(
    clean_dataset,
    test_df["dataset_idx"],
    processor,
    label_maps,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_batch,
)

print("Test samples:", len(test_dataset))


Test samples: 6611


## 8. Model Architectures

In [8]:
class ClassificationHead(nn.Module):
    def __init__(
        self,
        embedding_dim,
        num_classes,
        hidden_dim=512,
        dropout=0.2,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, features):
        return self.net(features)


class CLIPMultiTaskClassifier(nn.Module):
    def __init__(
        self,
        model_name,
        task_num_classes,
        hidden_dim=512,
        dropout=0.2,
    ):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(
            model_name
        )

        embedding_dim = (
            self.clip.config.projection_dim
        )

        self.heads = nn.ModuleDict(
            {
                task: ClassificationHead(
                    embedding_dim,
                    num_classes,
                    hidden_dim,
                    dropout,
                )
                for task, num_classes
                in task_num_classes.items()
            }
        )

    def forward(self, pixel_values):
        image_features = (
            self.clip.get_image_features(
                pixel_values=pixel_values
            )
        )

        image_features = F.normalize(
            image_features,
            dim=-1,
        )

        return {
            task: head(image_features)
            for task, head in self.heads.items()
        }


class CLIPMultiTaskClassifierV2(nn.Module):
    def __init__(
        self,
        model_name,
        task_num_classes,
        hidden_dim=512,
        dropout=0.2,
        color_feature_dim=37,
    ):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(
            model_name
        )

        embedding_dim = (
            self.clip.config.projection_dim
        )

        self.heads = nn.ModuleDict(
            {
                task: ClassificationHead(
                    embedding_dim,
                    num_classes,
                    hidden_dim,
                    dropout,
                )
                for task, num_classes
                in task_num_classes.items()
            }
        )

        self.master_to_sub = nn.Linear(
            task_num_classes["masterCategory"],
            task_num_classes["subCategory"],
            bias=False,
        )

        self.sub_to_article = nn.Linear(
            task_num_classes["subCategory"],
            task_num_classes["articleType"],
            bias=False,
        )

        self.article_to_season = nn.Linear(
            task_num_classes["articleType"],
            task_num_classes["season"],
            bias=False,
        )

        self.article_to_usage = nn.Linear(
            task_num_classes["articleType"],
            task_num_classes["usage"],
            bias=False,
        )

        self.color_branch = nn.Sequential(
            nn.LayerNorm(color_feature_dim),
            nn.Linear(color_feature_dim, 64),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(
                64,
                task_num_classes["baseColour"],
            ),
        )

    def forward(
        self,
        pixel_values,
        color_features,
    ):
        image_features = (
            self.clip.get_image_features(
                pixel_values=pixel_values
            )
        )

        image_features = F.normalize(
            image_features,
            dim=-1,
        )

        outputs = {
            task: head(image_features)
            for task, head in self.heads.items()
        }

        master_probs = torch.softmax(
            outputs["masterCategory"].detach(),
            dim=1,
        )

        outputs["subCategory"] = (
            outputs["subCategory"]
            + self.master_to_sub(master_probs)
        )

        sub_probs = torch.softmax(
            outputs["subCategory"].detach(),
            dim=1,
        )

        outputs["articleType"] = (
            outputs["articleType"]
            + self.sub_to_article(sub_probs)
        )

        article_probs = torch.softmax(
            outputs["articleType"].detach(),
            dim=1,
        )

        outputs["season"] = (
            outputs["season"]
            + self.article_to_season(article_probs)
        )

        outputs["usage"] = (
            outputs["usage"]
            + self.article_to_usage(article_probs)
        )

        outputs["baseColour"] = (
            outputs["baseColour"]
            + self.color_branch(color_features)
        )

        return outputs


## 9. Shared Metric Functions

In [9]:
def evaluate_predictions(
    y_true,
    y_pred,
    y_top3,
):
    task_metrics = {}

    for task in TASKS:
        top3_matches = [
            int(true_label)
            in set(map(int, top3_labels))
            for true_label, top3_labels
            in zip(
                y_true[task],
                y_top3[task],
            )
        ]

        task_metrics[task] = {
            "accuracy": float(
                accuracy_score(
                    y_true[task],
                    y_pred[task],
                )
            ),
            "macro_f1": float(
                f1_score(
                    y_true[task],
                    y_pred[task],
                    average="macro",
                    zero_division=0,
                )
            ),
            "top3_accuracy": float(
                np.mean(top3_matches)
            ),
        }

    exact_matches = np.ones(
        len(y_true[TASKS[0]]),
        dtype=bool,
    )

    for task in TASKS:
        exact_matches &= (
            np.asarray(y_true[task])
            == np.asarray(y_pred[task])
        )

    overall = {
        "average_accuracy": float(
            np.mean(
                [
                    task_metrics[task]["accuracy"]
                    for task in TASKS
                ]
            )
        ),
        "average_macro_f1": float(
            np.mean(
                [
                    task_metrics[task]["macro_f1"]
                    for task in TASKS
                ]
            )
        ),
        "average_top3_accuracy": float(
            np.mean(
                [
                    task_metrics[task]["top3_accuracy"]
                    for task in TASKS
                ]
            )
        ),
        "exact_match_accuracy": float(
            exact_matches.mean()
        ),
        "samples": int(len(exact_matches)),
    }

    return {
        "task_metrics": task_metrics,
        "overall_metrics": overall,
    }


@torch.inference_mode()
def collect_model_predictions(
    model,
    loader,
    device,
    uses_color_features,
):
    model.eval()

    y_true = {
        task: []
        for task in TASKS
    }

    y_pred = {
        task: []
        for task in TASKS
    }

    y_top3 = {
        task: []
        for task in TASKS
    }

    global_indices = []

    for batch in tqdm(
        loader,
        desc="Evaluating",
        leave=False,
    ):
        pixel_values = batch[
            "pixel_values"
        ].to(device)

        if uses_color_features:
            outputs = model(
                pixel_values,
                batch["color_features"].to(device),
            )
        else:
            outputs = model(pixel_values)

        for task in TASKS:
            probabilities = torch.softmax(
                outputs[task],
                dim=1,
            )

            top_k = min(
                3,
                probabilities.shape[1],
            )

            top_values, top_indices = torch.topk(
                probabilities,
                k=top_k,
                dim=1,
            )

            predictions = top_indices[:, 0]

            y_true[task].extend(
                batch["labels"][task]
                .numpy()
                .tolist()
            )

            y_pred[task].extend(
                predictions
                .cpu()
                .numpy()
                .tolist()
            )

            y_top3[task].extend(
                top_indices
                .cpu()
                .numpy()
                .tolist()
            )

        global_indices.extend(
            batch["global_indices"]
        )

    for task in TASKS:
        y_true[task] = np.asarray(
            y_true[task],
            dtype=np.int64,
        )

        y_pred[task] = np.asarray(
            y_pred[task],
            dtype=np.int64,
        )

        y_top3[task] = np.asarray(
            y_top3[task],
            dtype=np.int64,
        )

    return (
        y_true,
        y_pred,
        y_top3,
        global_indices,
    )


def get_state_dict(checkpoint):
    if "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]

    if "state_dict" in checkpoint:
        return checkpoint["state_dict"]

    return checkpoint


## 10. Majority Baseline

Top-1 uses the most frequent training label for each task.  
Top-3 uses the three most frequent training labels for each task.


In [10]:
majority_top1 = {}
majority_top3 = {}

for task in TASKS:
    counts = train_df[task].value_counts()

    majority_top1[task] = (
        label_maps[task]["label2id"][
            counts.index[0]
        ]
    )

    majority_top3[task] = [
        label_maps[task]["label2id"][label]
        for label in counts.index[:3]
    ]


majority_y_true = {
    task: test_df[task]
    .map(label_maps[task]["label2id"])
    .to_numpy(dtype=np.int64)
    for task in TASKS
}

majority_y_pred = {
    task: np.full(
        len(test_df),
        majority_top1[task],
        dtype=np.int64,
    )
    for task in TASKS
}

majority_y_top3 = {
    task: np.tile(
        np.asarray(
            majority_top3[task],
            dtype=np.int64,
        ),
        (len(test_df), 1),
    )
    for task in TASKS
}

majority_metrics = evaluate_predictions(
    majority_y_true,
    majority_y_pred,
    majority_y_top3,
)

print(
    json.dumps(
        majority_metrics["overall_metrics"],
        indent=2,
    )
)


{
  "average_accuracy": 0.42628087386822827,
  "average_macro_f1": 0.07927692465823664,
  "average_top3_accuracy": 0.7344469174752037,
  "exact_match_accuracy": 0.009529571925578581,
  "samples": 6611
}


In [11]:
def benchmark_majority_lookup(
    measured_runs=10000,
):
    start = time.perf_counter()

    for _ in range(measured_runs):
        _ = {
            task: majority_top1[task]
            for task in TASKS
        }

    elapsed_ms = (
        time.perf_counter() - start
    ) * 1000.0

    return elapsed_ms / measured_runs


majority_latency_ms = benchmark_majority_lookup()

print(
    "Majority lookup latency:",
    f"{majority_latency_ms:.6f} ms",
)


Majority lookup latency: 0.000565 ms


## 11. Evaluate Frozen CLIP + Heads (V1)

In [12]:
v1_checkpoint = v1_artifacts["checkpoint"]

v1_model_name = (
    v1_config.get("base_model_name")
    or v1_config.get("model_name")
    or v1_checkpoint.get("model_name")
    or MODEL_NAME
)

v1_hidden_dim = int(
    v1_config.get(
        "hidden_dim",
        v1_checkpoint.get(
            "hidden_dim",
            512,
        ),
    )
)

v1_dropout = float(
    v1_config.get(
        "dropout",
        v1_checkpoint.get(
            "dropout",
            0.2,
        ),
    )
)

v1_model = CLIPMultiTaskClassifier(
    model_name=v1_model_name,
    task_num_classes=task_num_classes,
    hidden_dim=v1_hidden_dim,
    dropout=v1_dropout,
)

v1_model.load_state_dict(
    get_state_dict(v1_checkpoint),
    strict=True,
)

v1_model.to(DEVICE)
v1_model.eval()

(
    v1_y_true,
    v1_y_pred,
    v1_y_top3,
    v1_global_indices,
) = collect_model_predictions(
    v1_model,
    test_loader,
    DEVICE,
    uses_color_features=False,
)

v1_metrics = evaluate_predictions(
    v1_y_true,
    v1_y_pred,
    v1_y_top3,
)

print(
    json.dumps(
        v1_metrics["overall_metrics"],
        indent=2,
    )
)


Evaluating:   0%|          | 0/104 [00:00<?, ?it/s]

{
  "average_accuracy": 0.8335026038852995,
  "average_macro_f1": 0.6568447666058456,
  "average_top3_accuracy": 0.9711087581303888,
  "exact_match_accuracy": 0.27938284677053393,
  "samples": 6611
}


## 12. Fair Latency Benchmark

In [13]:
@torch.inference_mode()
def benchmark_model_latency(
    model,
    sample,
    device,
    uses_color_features,
    warmup_runs=20,
    measured_runs=100,
):
    model.eval()

    pixel_values = sample[
        "pixel_values"
    ].unsqueeze(0).to(device)

    color_features = sample[
        "color_features"
    ].unsqueeze(0).to(device)

    for _ in range(warmup_runs):
        if uses_color_features:
            model(
                pixel_values,
                color_features,
            )
        else:
            model(pixel_values)

    if device.type == "cuda":
        torch.cuda.synchronize()

    times = []

    for _ in range(measured_runs):
        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        if uses_color_features:
            model(
                pixel_values,
                color_features,
            )
        else:
            model(pixel_values)

        if device.type == "cuda":
            torch.cuda.synchronize()

        times.append(
            (
                time.perf_counter() - start
            )
            * 1000.0
        )

    return {
        "average_ms": float(
            np.mean(times)
        ),
        "p50_ms": float(
            np.percentile(times, 50)
        ),
        "p95_ms": float(
            np.percentile(times, 95)
        ),
        "warmup_runs": warmup_runs,
        "measured_runs": measured_runs,
    }


latency_sample = test_dataset[0]

v1_latency = benchmark_model_latency(
    v1_model,
    latency_sample,
    DEVICE,
    uses_color_features=False,
    warmup_runs=LATENCY_WARMUP_RUNS,
    measured_runs=LATENCY_MEASURED_RUNS,
)

print(v1_latency)


{'average_ms': 6.226557110003341, 'p50_ms': 6.155085000045801, 'p95_ms': 6.70613664992743, 'warmup_runs': 20, 'measured_runs': 100}


## 13. Evaluate AutoCatalogAI V2

In [14]:
del v1_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


v2_checkpoint = v2_artifacts["checkpoint"]

v2_model = CLIPMultiTaskClassifierV2(
    model_name=(
        v2_checkpoint.get(
            "model_name",
            MODEL_NAME,
        )
    ),
    task_num_classes=(
        v2_checkpoint.get(
            "task_num_classes",
            task_num_classes,
        )
    ),
    hidden_dim=int(
        v2_checkpoint.get(
            "hidden_dim",
            HIDDEN_DIM,
        )
    ),
    dropout=float(
        v2_checkpoint.get(
            "dropout",
            DROPOUT,
        )
    ),
    color_feature_dim=int(
        v2_checkpoint.get(
            "color_feature_dim",
            COLOR_FEATURE_DIM,
        )
    ),
)

v2_model.load_state_dict(
    get_state_dict(v2_checkpoint),
    strict=True,
)

v2_model.to(DEVICE)
v2_model.eval()

(
    v2_y_true,
    v2_y_pred,
    v2_y_top3,
    v2_global_indices,
) = collect_model_predictions(
    v2_model,
    test_loader,
    DEVICE,
    uses_color_features=True,
)

v2_metrics = evaluate_predictions(
    v2_y_true,
    v2_y_pred,
    v2_y_top3,
)

v2_latency = benchmark_model_latency(
    v2_model,
    latency_sample,
    DEVICE,
    uses_color_features=True,
    warmup_runs=LATENCY_WARMUP_RUNS,
    measured_runs=LATENCY_MEASURED_RUNS,
)

print(
    json.dumps(
        v2_metrics["overall_metrics"],
        indent=2,
    )
)

print(v2_latency)


Evaluating:   0%|          | 0/104 [00:00<?, ?it/s]

{
  "average_accuracy": 0.8747758065561727,
  "average_macro_f1": 0.6740687827687263,
  "average_top3_accuracy": 0.981502690321326,
  "exact_match_accuracy": 0.4046286492209953,
  "samples": 6611
}
{'average_ms': 7.4737231199719645, 'p50_ms': 7.384413499949005, 'p95_ms': 8.047834450019309, 'warmup_runs': 20, 'measured_runs': 100}


## 14. Build the Comparison Table

The table uses **raw model predictions** for a fair comparison.  
V2 consistency correction is not applied here because the other baselines do not use post-processing.


In [16]:
comparison_rows = []


def add_comparison_row(
    model_name,
    metrics,
    latency_ms,
):
    overall = metrics["overall_metrics"]

    comparison_rows.append(
        {
            "Model": model_name,
            "Avg Accuracy": overall[
                "average_accuracy"
            ],
            "Avg Macro F1": overall[
                "average_macro_f1"
            ],
            "Top-3 Accuracy": overall[
                "average_top3_accuracy"
            ],
            "Exact Match": overall[
                "exact_match_accuracy"
            ],
            "Latency (ms)": latency_ms,
        }
    )


add_comparison_row(
    "Majority Baseline",
    majority_metrics,
    majority_latency_ms,
)

add_comparison_row(
    "Frozen CLIP + Heads (V1)",
    v1_metrics,
    v1_latency["average_ms"],
)

add_comparison_row(
    "AutoCatalogAI V2",
    v2_metrics,
    v2_latency["average_ms"],
)


comparison_df = pd.DataFrame(
    comparison_rows
)

display_df = comparison_df.copy()

for column in [
    "Avg Accuracy",
    "Avg Macro F1",
    "Top-3 Accuracy",
    "Exact Match",
]:
    display_df[column] = (
        display_df[column] * 100
    ).map(lambda value: f"{value:.2f}%")

display_df["Latency (ms)"] = (
    display_df["Latency (ms)"]
    .map(lambda value: f"{value:.3f}")
)

display(display_df)


,Model,Avg Accuracy,Avg Macro F1,Top-3 Accuracy,Exact Match,Latency (ms)
0,Majority Baseline,42.63%,7.93%,73.44%,0.95%,0.001
1,Frozen CLIP + Heads (V1),83.35%,65.68%,97.11%,27.94%,6.227
2,AutoCatalogAI V2,87.48%,67.41%,98.15%,40.46%,7.474


## 15. Per-Task Metrics

In [17]:
per_task_rows = []

models_and_metrics = {
    "Majority Baseline": majority_metrics,
    "Frozen CLIP + Heads (V1)": v1_metrics,
    "AutoCatalogAI V2": v2_metrics,
}

for model_name, metrics in models_and_metrics.items():
    for task in TASKS:
        task_result = metrics[
            "task_metrics"
        ][task]

        per_task_rows.append(
            {
                "Model": model_name,
                "Task": task,
                "Accuracy": task_result[
                    "accuracy"
                ],
                "Macro F1": task_result[
                    "macro_f1"
                ],
                "Top-3 Accuracy": task_result[
                    "top3_accuracy"
                ],
            }
        )


per_task_df = pd.DataFrame(
    per_task_rows
)

display(per_task_df)


,Model,Task,Accuracy,Macro F1,Top-3 Accuracy
0,Majority Baseline,gender,0.499168,0.133185,0.969899
1,Majority Baseline,masterCategory,0.484496,0.108790,0.948571
2,Majority Baseline,subCategory,0.349115,0.012939,0.584329
3,Majority Baseline,articleType,0.160339,0.002176,0.297837
4,Majority Baseline,baseColour,0.218726,0.008158,0.456209
5,Majority Baseline,season,0.489033,0.164212,0.939949
6,Majority Baseline,usage,0.783089,0.125479,0.944335
7,Frozen CLIP + Heads (V1),gender,0.888670,0.776600,0.995613
8,Frozen CLIP + Heads (V1),masterCategory,0.993344,0.862994,0.999395
9,Frozen CLIP + Heads (V1),subCategory,0.942974,0.710894,0.994706


## 16. Save Reproducibility Artifacts

In [18]:
comparison_csv_path = (
    OUTPUT_DIR
    / "model_comparison.csv"
)

per_task_csv_path = (
    OUTPUT_DIR
    / "model_comparison_per_task.csv"
)

comparison_json_path = (
    OUTPUT_DIR
    / "model_comparison.json"
)

metadata_path = (
    OUTPUT_DIR
    / "benchmark_metadata.json"
)

readme_table_path = (
    OUTPUT_DIR
    / "README_model_comparison.md"
)

comparison_df.to_csv(
    comparison_csv_path,
    index=False,
)

per_task_df.to_csv(
    per_task_csv_path,
    index=False,
)


comparison_payload = {
    row["Model"]: {
        "average_accuracy": float(
            row["Avg Accuracy"]
        ),
        "average_macro_f1": float(
            row["Avg Macro F1"]
        ),
        "average_top3_accuracy": float(
            row["Top-3 Accuracy"]
        ),
        "exact_match_accuracy": float(
            row["Exact Match"]
        ),
        "latency_ms": float(
            row["Latency (ms)"]
        ),
    }
    for row in comparison_rows
}

with open(
    comparison_json_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        comparison_payload,
        file,
        indent=2,
        ensure_ascii=False,
    )


benchmark_metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset_name": DATASET_NAME,
    "dataset_fingerprint": (
        clean_dataset._fingerprint
    ),
    "v1_repo_id": V1_REPO_ID,
    "v2_repo_id": V2_REPO_ID,
    "seed": SEED,
    "train_samples": int(
        len(train_df)
    ),
    "validation_samples": int(
        len(val_df)
    ),
    "test_samples": int(
        len(test_df)
    ),
    "test_split_sha256": (
        test_split_sha256
    ),
    "tasks": TASKS,
    "metric_policy": (
        "Raw model predictions for all rows"
    ),
    "latency_policy": {
        "batch_size": 1,
        "model_forward_only": True,
        "preprocessing_excluded": True,
        "warmup_runs": (
            LATENCY_WARMUP_RUNS
        ),
        "measured_runs": (
            LATENCY_MEASURED_RUNS
        ),
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": (
            transformers.__version__
        ),
        "device": str(DEVICE),
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "cuda_runtime": torch.version.cuda,
    },
}

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        benchmark_metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )


def markdown_percent(value):
    return f"{value * 100:.2f}%"


markdown_lines = [
    "| Model | Avg Accuracy | Avg Macro F1 | Top-3 Accuracy | Exact Match | Latency |",
    "|---|---:|---:|---:|---:|---:|",
]

for row in comparison_rows:
    latency_text = (
        f"{row['Latency (ms)']:.3f} ms"
    )

    markdown_lines.append(
        "| "
        + " | ".join(
            [
                row["Model"],
                markdown_percent(
                    row["Avg Accuracy"]
                ),
                markdown_percent(
                    row["Avg Macro F1"]
                ),
                markdown_percent(
                    row["Top-3 Accuracy"]
                ),
                markdown_percent(
                    row["Exact Match"]
                ),
                latency_text,
            ]
        )
        + " |"
    )

markdown_lines.extend(
    [
        "",
        (
            "> All models were evaluated on the same "
            f"{len(test_df):,}-image held-out test split. "
            "Metrics use raw model predictions. "
            "Latency is batch-size-1 model-forward time "
            "with preprocessing excluded."
        ),
    ]
)

readme_markdown = "\n".join(
    markdown_lines
)

readme_table_path.write_text(
    readme_markdown,
    encoding="utf-8",
)

print(readme_markdown)

print("\nSaved files:")

for path in sorted(
    OUTPUT_DIR.iterdir()
):
    print("-", path)


| Model | Avg Accuracy | Avg Macro F1 | Top-3 Accuracy | Exact Match | Latency |
|---|---:|---:|---:|---:|---:|
| Majority Baseline | 42.63% | 7.93% | 73.44% | 0.95% | 0.001 ms |
| Frozen CLIP + Heads (V1) | 83.35% | 65.68% | 97.11% | 27.94% | 6.227 ms |
| AutoCatalogAI V2 | 87.48% | 67.41% | 98.15% | 40.46% | 7.474 ms |

> All models were evaluated on the same 6,611-image held-out test split. Metrics use raw model predictions. Latency is batch-size-1 model-forward time with preprocessing excluded.

Saved files:
- artifacts/evaluation/model_comparison/README_model_comparison.md
- artifacts/evaluation/model_comparison/benchmark_metadata.json
- artifacts/evaluation/model_comparison/model_comparison.csv
- artifacts/evaluation/model_comparison/model_comparison.json
- artifacts/evaluation/model_comparison/model_comparison_per_task.csv
